# `text_embedding_0609.ipynb`

目的：建立 **單篇文本部署推論 benchmark**，避免把整批訓練、整批斷詞、整批特徵建構時間誤當成單篇上線推論時間。

這份 notebook 會做三件事：
1. 依照 `text_embedding_0224.ipynb` 的同一份資料與切分方式重新訓練模型。
2. 保留每個組合的已訓練前處理器與分類器。
3. 逐篇量測 test set 在部署情境下的 `tokenize_sec`、`feature_sec`、`predict_sec`、`total_infer_sec`。

輸出檔案：
- `results_model_matrix_0609.csv`
- `results_model_inference_detail_0609.csv`
- `results_model_inference_summary_0609.csv`


In [7]:
from pathlib import Path
import os
import re
import io
import time
import json
import contextlib
import sqlite3

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
from IPython.display import display

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False


## 1. 參數設定

- 若你只是先測流程，可先把 `BENCHMARK_TEXT_LIMIT` 設成 `30`。
- 若你要正式論文數字，建議 `BENCHMARK_TEXT_LIMIT = None`，完整跑完整個 test set。
- `BENCHMARK_REPEATS` 用來減少單次波動；正式版建議 `3`。


In [8]:
BASE_DIR = Path.cwd()
if not (BASE_DIR / 'results_model_matrix.csv').exists():
    BASE_DIR = Path(r'd:/NTPU_class/paper/code')

DB_FILE = Path(r'd:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite')
OUT_MATRIX = BASE_DIR / 'results_model_matrix_0609.csv'
OUT_DETAIL = BASE_DIR / 'results_model_inference_detail_0609.csv'
OUT_SUMMARY = BASE_DIR / 'results_model_inference_summary_0609.csv'

RANDOM_SEED = 42
TEST_SIZE = 0.20
TEXT_COL = 'content'
Y_COL = 'y_llm'

TFIDF_MAX_FEATURES = 20000
TFIDF_MIN_DF = 5
TFIDF_NGRAM = (1, 2)
SVD_DIM = 300
EMB_DIM = 200
W2V_EPOCHS = 10
FT_EPOCHS = 10
N_WORKERS = max(1, (os.cpu_count() or 4) - 1)

RUN_TOKENIZERS = ['jieba', 'ckip']
RUN_REPRS = ['tfidf', f'tfidf_svd{SVD_DIM}', 'w2v_mean', 'fasttext_mean']
RUN_MODELS = ['lr', 'svm_rbf', 'svm_poly', 'rf', 'mlp', 'xgb']

BENCHMARK_TEXT_LIMIT = None
BENCHMARK_REPEATS = 3
WARMUP_TEXTS = 3


## 2. 載入與清理資料

這裡沿用 `text_embedding_0224.ipynb` 的資料來源與欄位設定。


In [9]:
sql = """
WITH latest AS (
  SELECT
    article_id,
    MAX(created_at) AS max_created_at
  FROM eb_annotations
  WHERE level = 1
    AND article_id IS NOT NULL
  GROUP BY article_id
)
SELECT
  a.id          AS article_id,
  a.link_id     AS link_id,
  a.url         AS url,
  a.title       AS title,
  a.post_time   AS post_time,
  a.content     AS content,
  a.word_count  AS word_count,
  a.reply_count AS reply_count,
  l.keyword     AS keyword,
  l.has_emotional_abuse AS y_rule,
  e.score_overall         AS y_llm_score,
  e.confidence            AS llm_confidence,
  e.is_eb_rule            AS y_rule_calc,
  e.is_eb_llm             AS y_llm,
  e.is_eb_llm_confidence  AS y_llm_confidence,
  e.main_strategy,
  e.main_strategy_detail,
  e.pua_source,
  e.model_name,
  e.model_version,
  e.knowledge_base,
  e.created_at            AS anno_created_at
FROM articles a
JOIN links l
  ON a.link_id = l.id
LEFT JOIN latest t
  ON a.id = t.article_id
LEFT JOIN eb_annotations e
  ON e.article_id = t.article_id
 AND e.created_at = t.max_created_at
 AND e.level = 1
WHERE a.content IS NOT NULL
  AND TRIM(a.content) <> '';
"""

with sqlite3.connect(DB_FILE) as conn:
    df = pd.read_sql_query(sql, conn)

df[Y_COL] = pd.to_numeric(df[Y_COL], errors='coerce')
df = df.dropna(subset=[TEXT_COL, Y_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str)
df[Y_COL] = df[Y_COL].astype(int)

display(df.head())
print('資料筆數 =', len(df))
print(df[Y_COL].value_counts())


,article_id,link_id,url,title,post_time,content,word_count,reply_count,keyword,y_rule,...,y_rule_calc,y_llm,y_llm_confidence,main_strategy,main_strategy_detail,pua_source,model_name,model_version,knowledge_base,anno_created_at
0,68fce9b6af1137205015fd06,68f87edcaf1137700cb26e94,https://www.mobile01.com/topicdetail.php?f=330...,被爸媽情緒勒索很痛苦 心理諮商有效嗎?,2024-12-10 9:53,大家都遇過父母情緒勒索嗎？從小我就是一個很聽話的小孩，因為我爸媽控制慾很重，如果不聽他們的話...,397,8,情緒勒索,1,...,1.0,1,0.9,power/emotion/blame,利用情感、控制慾和責任感來勒索子女,family,mistral,latest,pua_db,2026-02-10 09:38:45
1,68fce9f3af11377624f11eda,68f87edcaf1137700cb26e95,https://www.mobile01.com/topicdetail.php?f=292...,看完這篇其實 情緒勒索是不是也等於控制慾強?,2021-06-28 8:25,看完這篇其實 情緒勒索是不是也等於控制慾強?\n\n剛好在兩性版看到一段話 突然對控制慾強這...,2106,7,情緒勒索,1,...,1.0,1,1.0,power,使用憤怒和控制以獲得屌服的對象，並利用文字紀錄來強調自己的正確性,partner,mistral,latest,pua_db,2026-02-10 09:38:49
2,68fcea01af11377624f11edb,68f87edcaf1137700cb26e96,https://www.mobile01.com/topicdetail.php?f=37&...,沒有情緒勒索，只有忠言逆耳,2023-03-08 12:09,現在人(尤其是年輕人)遇到不好聽的話，就會說在情緒勒索他們\n實際上並不存在情緒勒索這種東西...,143,8,情緒勒索,1,...,1.0,0,1.0,emotion,利用反舉與責罪的方式，挑戰受眾的情感層面,online,mistral,latest,pua_db,2026-02-10 09:38:51
3,68fcea15af11377624f11edc,68f87edcaf1137700cb26e98,https://www.mobile01.com/topicdetail.php?f=594...,#書籍推薦 #不被情緒勒索的51個方法,2022-11-15 10:31,"(時間會證明,當初自己是對的...也會證明當初自己是錯的 ...?)\n(所以才需要每天寫日...",1248,0,情緒勒索,1,...,1.0,1,0.9,emotion,Using self-reflection and self-improvement tec...,self,mistral,latest,pua_db,2026-02-10 09:38:55
4,68fcea6eaf11379f9c929394,68f87edcaf1137700cb26e9c,https://www.mobile01.com/topicdetail.php?f=292...,有關情緒勒索,2023-09-07 16:49,請問各位，我有個疑問\n如果只是問另一半要不要吃飯\n或著是問一起出去玩好不好，\n又或者問...,63,8,情緒勒索,1,...,0.0,0,1.0,none,無情緒勒索，而是正常的互動或問題提出,online,mistral,latest,pua_db,2026-02-10 09:38:57


資料筆數 = 1385
y_llm
1    1105
0     280
Name: count, dtype: int64


In [10]:
print("文本數:", len(df))
print("總字數:", df["content"].str.len().sum())
print("平均字數:", df["content"].str.len().mean())
print("最大字數:", df["content"].str.len().max())
print("最小字數:", df["content"].str.len().min())

文本數: 1385
總字數: 295279
平均字數: 213.19783393501805
最大字數: 2353
最小字數: 2


## 3. 切分資料

使用與原 notebook 相同的固定 random seed 與 stratified split。


In [11]:
df_train, df_test = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df[Y_COL]
)

y_train = df_train[Y_COL].values
y_test = df_test[Y_COL].values

if BENCHMARK_TEXT_LIMIT is not None:
    df_test_bench = df_test.head(BENCHMARK_TEXT_LIMIT).copy()
else:
    df_test_bench = df_test.copy()

print('train =', len(df_train))
print('test =', len(df_test))
print('benchmark texts =', len(df_test_bench))


train = 1108
test = 277
benchmark texts = 277


## 4. 斷詞、特徵、模型工具


In [12]:
import jieba


def normalize_text(text: str) -> str:
    if text is None:
        return ''
    return re.sub(r'\s+', ' ', str(text)).strip()


def tokenize_jieba(text: str):
    text = normalize_text(text)
    if not text:
        return []
    return [t for t in jieba.cut(text, cut_all=False) if t.strip()]


_CKIP_WS = None

def get_ckip_ws():
    global _CKIP_WS
    if _CKIP_WS is None:
        from ckip_transformers.nlp import CkipWordSegmenter
        f_out, f_err = io.StringIO(), io.StringIO()
        with contextlib.redirect_stdout(f_out), contextlib.redirect_stderr(f_err):
            _CKIP_WS = CkipWordSegmenter(model='bert-base')
    return _CKIP_WS


def tokenize_ckip(text: str):
    text = normalize_text(text)
    if not text:
        return []
    ws = get_ckip_ws()
    f_out, f_err = io.StringIO(), io.StringIO()
    with contextlib.redirect_stdout(f_out), contextlib.redirect_stderr(f_err):
        tokens = ws([text])[0]
    return [t for t in tokens if t.strip()]


def tokens_to_corpus(tokens):
    return ' '.join(tokens)


def make_tokens(df_part, text_col, tokenizer_fn):
    return [tokenizer_fn(text) for text in df_part[text_col].astype(str).tolist()]


def mean_pool_single(tokens, wv, vector_size):
    vecs = [wv[w] for w in tokens if w in wv]
    if vecs:
        return np.mean(vecs, axis=0, dtype=np.float32).reshape(1, -1)
    return np.zeros((1, vector_size), dtype=np.float32)


def mean_pool_batch(tokens_list, wv, vector_size):
    X = np.zeros((len(tokens_list), vector_size), dtype=np.float32)
    cov = []
    for i, toks in enumerate(tokens_list):
        vecs = [wv[w] for w in toks if w in wv]
        if vecs:
            X[i] = np.mean(vecs, axis=0)
            cov.append(len(vecs) / max(len(toks), 1))
        else:
            cov.append(0.0)
    return X, float(np.mean(cov))


def build_models(seed=RANDOM_SEED):
    models = {}
    models['lr'] = LogisticRegression(max_iter=2000, solver='liblinear', class_weight='balanced')
    models['svm_rbf'] = SVC(kernel='rbf', probability=True, class_weight='balanced')
    models['svm_poly'] = SVC(kernel='poly', degree=3, probability=True, class_weight='balanced')
    models['rf'] = RandomForestClassifier(
        n_estimators=600,
        random_state=seed,
        n_jobs=-1,
        class_weight='balanced_subsample'
    )
    if HAS_XGB:
        models['xgb'] = XGBClassifier(
            n_estimators=600,
            max_depth=6,
            learning_rate=0.08,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=seed,
            n_jobs=-1,
        )
    models['mlp'] = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=40,
        early_stopping=True,
        random_state=seed,
    )
    return models


def eval_binary(y_true, pred, score):
    return {
        'acc': accuracy_score(y_true, pred),
        'f1': f1_score(y_true, pred),
        'roc_auc': roc_auc_score(y_true, score) if score is not None else np.nan,
        'pr_auc': average_precision_score(y_true, score) if score is not None else np.nan,
    }


## 5. 訓練所有組合，並保留部署需要的 artifacts

這一步仍然會花時間，因為它會重新訓練模型；但後面的 benchmark 只量單篇推論，不把訓練時間算進去。


In [13]:
from gensim.models import Word2Vec, FastText


def fit_estimator(model, Xtr, ytr, need_scale=False):
    t0 = time.perf_counter()
    if need_scale:
        fitted = Pipeline([('scaler', StandardScaler()), ('clf', model)])
        fitted.fit(Xtr, ytr)
    else:
        fitted = model
        fitted.fit(Xtr, ytr)
    train_sec = time.perf_counter() - t0
    return fitted, train_sec


def predict_eval(fitted, Xte, yte):
    t0 = time.perf_counter()
    pred = fitted.predict(Xte)
    score = fitted.predict_proba(Xte)[:, 1] if hasattr(fitted, 'predict_proba') else None
    test_pred_sec = time.perf_counter() - t0
    met = eval_binary(yte, pred, score)
    met['test_pred_sec'] = test_pred_sec
    return met


def train_bundles(df_train, df_test, y_train, y_test):
    tokenize_map = {
        'jieba': tokenize_jieba,
        'ckip': tokenize_ckip,
    }
    tokenize_map = {k: v for k, v in tokenize_map.items() if k in RUN_TOKENIZERS}

    model_templates = build_models(seed=RANDOM_SEED)
    run_models = [m for m in RUN_MODELS if m in model_templates]

    bundle_rows = []
    bundles = {}

    for tok_name, tok_fn in tokenize_map.items():
        t0 = time.perf_counter()
        tok_tr = make_tokens(df_train, TEXT_COL, tok_fn)
        tok_te = make_tokens(df_test, TEXT_COL, tok_fn)
        prep_sec = time.perf_counter() - t0

        corpus_tr = [tokens_to_corpus(t) for t in tok_tr]
        corpus_te = [tokens_to_corpus(t) for t in tok_te]

        if 'tfidf' in RUN_REPRS or f'tfidf_svd{SVD_DIM}' in RUN_REPRS:
            t0 = time.perf_counter()
            tfidf_vec = TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                min_df=TFIDF_MIN_DF,
                ngram_range=TFIDF_NGRAM,
                dtype=np.float32,
            )
            Xtr_tfidf = tfidf_vec.fit_transform(corpus_tr)
            Xte_tfidf = tfidf_vec.transform(corpus_te)
            tfidf_sec = time.perf_counter() - t0

            if 'tfidf' in RUN_REPRS and 'lr' in run_models:
                fitted, train_sec = fit_estimator(build_models(RANDOM_SEED)['lr'], Xtr_tfidf, y_train, need_scale=False)
                met = predict_eval(fitted, Xte_tfidf, y_test)
                key = (tok_name, 'tfidf', 'lr')
                bundles[key] = {
                    'tokenizer_name': tok_name,
                    'tokenizer_fn': tok_fn,
                    'repr': 'tfidf',
                    'model': 'lr',
                    'feature_artifact': {'vectorizer': tfidf_vec},
                    'estimator': fitted,
                    'need_scale': False,
                }
                bundle_rows.append({
                    'tokenizer': tok_name, 'repr': 'tfidf', 'model': 'lr',
                    'prep_sec': prep_sec, 'feat_sec': tfidf_sec, 'train_sec': train_sec,
                    **met,
                })

            if f'tfidf_svd{SVD_DIM}' in RUN_REPRS:
                t0 = time.perf_counter()
                svd = TruncatedSVD(n_components=SVD_DIM, random_state=RANDOM_SEED)
                Xtr_svd = svd.fit_transform(Xtr_tfidf)
                Xte_svd = svd.transform(Xte_tfidf)
                svd_sec = time.perf_counter() - t0
                for mname in [m for m in run_models if m != 'lr']:
                    need_scale = mname.startswith('svm') or mname == 'mlp'
                    fitted, train_sec = fit_estimator(build_models(RANDOM_SEED)[mname], Xtr_svd, y_train, need_scale=need_scale)
                    met = predict_eval(fitted, Xte_svd, y_test)
                    key = (tok_name, f'tfidf_svd{SVD_DIM}', mname)
                    bundles[key] = {
                        'tokenizer_name': tok_name,
                        'tokenizer_fn': tok_fn,
                        'repr': f'tfidf_svd{SVD_DIM}',
                        'model': mname,
                        'feature_artifact': {'vectorizer': tfidf_vec, 'svd': svd},
                        'estimator': fitted,
                        'need_scale': need_scale,
                    }
                    bundle_rows.append({
                        'tokenizer': tok_name, 'repr': f'tfidf_svd{SVD_DIM}', 'model': mname,
                        'prep_sec': prep_sec, 'feat_sec': tfidf_sec + svd_sec, 'train_sec': train_sec,
                        **met,
                    })

        if 'w2v_mean' in RUN_REPRS:
            t0 = time.perf_counter()
            w2v_model = Word2Vec(
                sentences=tok_tr,
                vector_size=EMB_DIM,
                window=5,
                min_count=2,
                workers=N_WORKERS,
            )
            w2v_model.train(tok_tr, total_examples=len(tok_tr), epochs=W2V_EPOCHS)
            Xtr_w2v, cov_tr = mean_pool_batch(tok_tr, w2v_model.wv, EMB_DIM)
            Xte_w2v, cov_te = mean_pool_batch(tok_te, w2v_model.wv, EMB_DIM)
            w2v_sec = time.perf_counter() - t0

            for mname in run_models:
                need_scale = mname.startswith('svm') or mname in ['lr', 'mlp']
                fitted, train_sec = fit_estimator(build_models(RANDOM_SEED)[mname], Xtr_w2v, y_train, need_scale=need_scale)
                met = predict_eval(fitted, Xte_w2v, y_test)
                key = (tok_name, 'w2v_mean', mname)
                bundles[key] = {
                    'tokenizer_name': tok_name,
                    'tokenizer_fn': tok_fn,
                    'repr': 'w2v_mean',
                    'model': mname,
                    'feature_artifact': {'wv': w2v_model.wv, 'vector_size': EMB_DIM},
                    'estimator': fitted,
                    'need_scale': need_scale,
                }
                bundle_rows.append({
                    'tokenizer': tok_name, 'repr': 'w2v_mean', 'model': mname,
                    'cov_train': cov_tr, 'cov_test': cov_te,
                    'prep_sec': prep_sec, 'feat_sec': w2v_sec, 'train_sec': train_sec,
                    **met,
                })

        if 'fasttext_mean' in RUN_REPRS:
            t0 = time.perf_counter()
            ft_model = FastText(
                sentences=tok_tr,
                vector_size=EMB_DIM,
                window=5,
                min_count=2,
                workers=N_WORKERS,
            )
            ft_model.train(tok_tr, total_examples=len(tok_tr), epochs=FT_EPOCHS)
            Xtr_ft, cov_tr = mean_pool_batch(tok_tr, ft_model.wv, EMB_DIM)
            Xte_ft, cov_te = mean_pool_batch(tok_te, ft_model.wv, EMB_DIM)
            ft_sec = time.perf_counter() - t0

            for mname in run_models:
                need_scale = mname.startswith('svm') or mname in ['lr', 'mlp']
                fitted, train_sec = fit_estimator(build_models(RANDOM_SEED)[mname], Xtr_ft, y_train, need_scale=need_scale)
                met = predict_eval(fitted, Xte_ft, y_test)
                key = (tok_name, 'fasttext_mean', mname)
                bundles[key] = {
                    'tokenizer_name': tok_name,
                    'tokenizer_fn': tok_fn,
                    'repr': 'fasttext_mean',
                    'model': mname,
                    'feature_artifact': {'wv': ft_model.wv, 'vector_size': EMB_DIM},
                    'estimator': fitted,
                    'need_scale': need_scale,
                }
                bundle_rows.append({
                    'tokenizer': tok_name, 'repr': 'fasttext_mean', 'model': mname,
                    'cov_train': cov_tr, 'cov_test': cov_te,
                    'prep_sec': prep_sec, 'feat_sec': ft_sec, 'train_sec': train_sec,
                    **met,
                })

    df_matrix = pd.DataFrame(bundle_rows)
    df_matrix['total_sec'] = df_matrix['prep_sec'] + df_matrix['feat_sec'] + df_matrix['train_sec'] + df_matrix['test_pred_sec']

    for col in ['acc', 'f1', 'roc_auc', 'pr_auc', 'prep_sec', 'feat_sec', 'train_sec', 'test_pred_sec', 'total_sec', 'cov_train', 'cov_test']:
        if col in df_matrix.columns:
            df_matrix[col] = pd.to_numeric(df_matrix[col], errors='coerce')

    df_matrix = df_matrix.sort_values(['pr_auc', 'roc_auc', 'f1'], ascending=False).reset_index(drop=True)
    return df_matrix, bundles


df_matrix, bundles = train_bundles(df_train, df_test, y_train, y_test)
display(df_matrix.head(20))
print('bundle count =', len(bundles))


BertForTokenClassification LOAD REPORT from: ckiplab/bert-base-chinese-ws
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,tokenizer,repr,model,prep_sec,feat_sec,train_sec,acc,f1,roc_auc,pr_auc,test_pred_sec,cov_train,cov_test,total_sec
0,ckip,fasttext_mean,svm_rbf,204.058941,3.432719,0.188101,0.700361,0.770083,0.903604,0.975405,0.029079,1.000000,1.000000,207.708839
1,jieba,tfidf,lr,1.578999,0.445531,0.003299,0.801444,0.862843,0.901745,0.975289,0.000464,NaN,NaN,2.028293
2,jieba,w2v_mean,svm_rbf,1.578999,1.510796,0.204456,0.747292,0.813830,0.899685,0.975225,0.035495,0.934645,0.898995,3.329746
3,ckip,tfidf_svd300,xgb,204.058941,0.710974,2.057627,0.851986,0.908686,0.898109,0.975032,0.004341,NaN,NaN,206.831882
4,ckip,fasttext_mean,mlp,204.058941,3.432719,0.119805,0.855596,0.912664,0.900695,0.974641,0.001813,1.000000,1.000000,207.613277
5,ckip,w2v_mean,svm_rbf,204.058941,1.279816,0.194388,0.736462,0.805333,0.899240,0.974485,0.030795,0.953589,0.919307,205.563939
6,ckip,w2v_mean,svm_poly,204.058941,1.279816,0.141978,0.722022,0.791328,0.898109,0.974295,0.011337,0.953589,0.919307,205.492071
7,ckip,fasttext_mean,lr,204.058941,3.432719,0.033527,0.765343,0.831169,0.900856,0.974289,0.001262,1.000000,1.000000,207.526448
8,ckip,w2v_mean,lr,204.058941,1.279816,0.048457,0.776173,0.842640,0.899887,0.974257,0.001407,0.953589,0.919307,205.388620
9,ckip,tfidf,lr,204.058941,0.264325,0.003503,0.812274,0.870647,0.897786,0.973989,0.000389,NaN,NaN,204.327158


bundle count = 36


## 6. 定義單篇部署推論 benchmark

這裡的 `total_infer_sec` = `tokenize_sec + feature_sec + predict_sec`。

注意：
- **不包含訓練時間**
- **不包含整批預先處理時間**
- 只模擬「單篇新文章進來後」的部署推論成本


In [14]:
def featurize_single(tokens, bundle):
    repr_name = bundle['repr']
    artifact = bundle['feature_artifact']

    t0 = time.perf_counter()
    if repr_name == 'tfidf':
        X = artifact['vectorizer'].transform([tokens_to_corpus(tokens)])
    elif repr_name.startswith('tfidf_svd'):
        X_tfidf = artifact['vectorizer'].transform([tokens_to_corpus(tokens)])
        X = artifact['svd'].transform(X_tfidf)
    elif repr_name in ['w2v_mean', 'fasttext_mean']:
        X = mean_pool_single(tokens, artifact['wv'], artifact['vector_size'])
    else:
        raise ValueError(f'Unknown repr: {repr_name}')
    feature_sec = time.perf_counter() - t0
    return X, feature_sec


def predict_single(X, bundle):
    est = bundle['estimator']
    t0 = time.perf_counter()
    pred = est.predict(X)[0]
    score = est.predict_proba(X)[0, 1] if hasattr(est, 'predict_proba') else np.nan
    predict_sec = time.perf_counter() - t0
    return pred, score, predict_sec


def benchmark_single_text(text, bundle, repeats=1):
    tokenizer_fn = bundle['tokenizer_fn']
    metrics = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        tokens = tokenizer_fn(text)
        tokenize_sec = time.perf_counter() - t0

        X, feature_sec = featurize_single(tokens, bundle)
        pred, score, predict_sec = predict_single(X, bundle)
        total_infer_sec = tokenize_sec + feature_sec + predict_sec

        metrics.append({
            'token_count': len(tokens),
            'pred': int(pred),
            'score': float(score) if score == score else np.nan,
            'tokenize_sec': tokenize_sec,
            'feature_sec': feature_sec,
            'predict_sec': predict_sec,
            'total_infer_sec': total_infer_sec,
        })
    return pd.DataFrame(metrics)


## 7. 先 warm-up，再跑 test set benchmark

- warm-up 主要避免第一次載入模型或 tokenizer 時的初始化噪音。
- `results_model_inference_detail_0609.csv` 會保留每篇每次量測明細。
- `results_model_inference_summary_0609.csv` 會彙整成平均值、中位數、P95 等統計。


In [15]:
# warm-up
warmup_texts = df_test_bench[TEXT_COL].astype(str).head(WARMUP_TEXTS).tolist()
for text in warmup_texts:
    for bundle in bundles.values():
        _ = benchmark_single_text(text, bundle, repeats=1)

print('warm-up done')


warm-up done


In [16]:
detail_rows = []

for row_idx, row in enumerate(df_test_bench.itertuples(index=False), start=1):
    text = getattr(row, TEXT_COL)
    article_id = getattr(row, 'article_id', None)
    true_y = getattr(row, Y_COL)
    title = getattr(row, 'title', '')

    for (tok_name, repr_name, model_name), bundle in bundles.items():
        bench_df = benchmark_single_text(text, bundle, repeats=BENCHMARK_REPEATS)
        bench_avg = bench_df.mean(numeric_only=True)

        detail_rows.append({
            'article_id': article_id,
            'title': title,
            'y_true': true_y,
            'text_length': len(str(text)),
            'tokenizer': tok_name,
            'repr': repr_name,
            'model': model_name,
            'repeats': BENCHMARK_REPEATS,
            'token_count_avg': bench_avg.get('token_count', np.nan),
            'pred_avg': bench_avg.get('pred', np.nan),
            'score_avg': bench_avg.get('score', np.nan),
            'tokenize_sec': bench_avg.get('tokenize_sec', np.nan),
            'feature_sec': bench_avg.get('feature_sec', np.nan),
            'predict_sec': bench_avg.get('predict_sec', np.nan),
            'total_infer_sec': bench_avg.get('total_infer_sec', np.nan),
        })

    if row_idx % 20 == 0:
        print(f'processed {row_idx}/{len(df_test_bench)} texts')

df_detail = pd.DataFrame(detail_rows)
display(df_detail.head())
print('detail rows =', len(df_detail))


processed 20/277 texts
processed 40/277 texts
processed 60/277 texts
processed 80/277 texts
processed 100/277 texts
processed 120/277 texts
processed 140/277 texts
processed 160/277 texts
processed 180/277 texts
processed 200/277 texts
processed 220/277 texts
processed 240/277 texts
processed 260/277 texts


,article_id,title,y_true,text_length,tokenizer,repr,model,repeats,token_count_avg,pred_avg,score_avg,tokenize_sec,feature_sec,predict_sec,total_infer_sec
0,691074a0dbeaa9b9eebdf9e2,黑人吃定大牙沒證據,1,5,jieba,tfidf,lr,3,4.0,1.0,0.983642,0.000095,0.000864,0.000576,0.001536
1,691074a0dbeaa9b9eebdf9e2,黑人吃定大牙沒證據,1,5,jieba,tfidf_svd300,svm_rbf,3,4.0,1.0,0.982858,0.000108,0.002647,0.002261,0.005016
2,691074a0dbeaa9b9eebdf9e2,黑人吃定大牙沒證據,1,5,jieba,tfidf_svd300,svm_poly,3,4.0,1.0,0.840443,0.000099,0.002626,0.002099,0.004824
3,691074a0dbeaa9b9eebdf9e2,黑人吃定大牙沒證據,1,5,jieba,tfidf_svd300,rf,3,4.0,1.0,1.000000,0.000079,0.002229,0.198961,0.201269
4,691074a0dbeaa9b9eebdf9e2,黑人吃定大牙沒證據,1,5,jieba,tfidf_svd300,mlp,3,4.0,1.0,0.863553,0.000069,0.001824,0.000824,0.002717


detail rows = 9972


## 8. 彙整單篇推論統計


In [17]:
def p95(series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    return np.percentile(s, 95) if len(s) else np.nan

summary_rows = []
for keys, part in df_detail.groupby(['tokenizer', 'repr', 'model'], dropna=False):
    tok_name, repr_name, model_name = keys
    summary_rows.append({
        'tokenizer': tok_name,
        'repr': repr_name,
        'model': model_name,
        'n_texts': len(part),
        'tokenize_mean_sec': part['tokenize_sec'].mean(),
        'feature_mean_sec': part['feature_sec'].mean(),
        'predict_mean_sec': part['predict_sec'].mean(),
        'total_infer_mean_sec': part['total_infer_sec'].mean(),
        'total_infer_median_sec': part['total_infer_sec'].median(),
        'total_infer_min_sec': part['total_infer_sec'].min(),
        'total_infer_max_sec': part['total_infer_sec'].max(),
        'total_infer_p95_sec': p95(part['total_infer_sec']),
        'text_length_mean': part['text_length'].mean(),
        'token_count_mean': part['token_count_avg'].mean(),
    })

df_summary = pd.DataFrame(summary_rows)
df_summary = df_summary.sort_values(['total_infer_mean_sec', 'total_infer_median_sec', 'tokenize_mean_sec']).reset_index(drop=True)
display(df_summary.head(20))


,tokenizer,repr,model,n_texts,tokenize_mean_sec,feature_mean_sec,predict_mean_sec,total_infer_mean_sec,total_infer_median_sec,total_infer_min_sec,total_infer_max_sec,total_infer_p95_sec,text_length_mean,token_count_mean
0,jieba,w2v_mean,mlp,277,0.001245,0.000277,0.000670,0.002192,0.001223,0.000601,0.015637,0.007305,216.592058,122.862816
1,jieba,fasttext_mean,mlp,277,0.001243,0.000475,0.000671,0.002390,0.001237,0.000616,0.018764,0.008208,216.592058,122.862816
2,jieba,w2v_mean,lr,277,0.001971,0.000413,0.000820,0.003204,0.001468,0.000597,0.031012,0.011423,216.592058,122.862816
3,jieba,w2v_mean,svm_poly,277,0.001781,0.000376,0.001055,0.003212,0.001754,0.000772,0.016918,0.010637,216.592058,122.862816
4,jieba,tfidf,lr,277,0.002117,0.000857,0.000374,0.003349,0.001933,0.000782,0.021140,0.010912,216.592058,122.862816
5,jieba,w2v_mean,svm_rbf,277,0.001897,0.000401,0.001166,0.003463,0.001867,0.000880,0.018726,0.011553,216.592058,122.862816
6,jieba,tfidf_svd300,mlp,277,0.001270,0.001702,0.000714,0.003686,0.002791,0.001895,0.016458,0.008452,216.592058,122.862816
7,jieba,fasttext_mean,svm_poly,277,0.001872,0.000720,0.001152,0.003744,0.002040,0.000767,0.019665,0.011361,216.592058,122.862816
8,jieba,fasttext_mean,lr,277,0.002154,0.000875,0.000946,0.003975,0.001759,0.000634,0.034693,0.014239,216.592058,122.862816
9,jieba,fasttext_mean,svm_rbf,277,0.002088,0.000818,0.001288,0.004194,0.002061,0.000823,0.021711,0.014875,216.592058,122.862816


## 9. 合併效能與部署推論時間

這一張表後續可以直接拿去接 `plotly_0609.ipynb`。


In [18]:
df_matrix_merge = df_matrix.merge(
    df_summary,
    on=['tokenizer', 'repr', 'model'],
    how='left'
)

df_matrix_merge.to_csv(OUT_MATRIX, index=False, encoding='utf-8-sig')
df_detail.to_csv(OUT_DETAIL, index=False, encoding='utf-8-sig')
df_summary.to_csv(OUT_SUMMARY, index=False, encoding='utf-8-sig')

print('saved:')
print('-', OUT_MATRIX)
print('-', OUT_DETAIL)
print('-', OUT_SUMMARY)

display(df_matrix_merge.head(20))


saved:
- d:\NTPU_class\paper\code\results_model_matrix_0609.csv
- d:\NTPU_class\paper\code\results_model_inference_detail_0609.csv
- d:\NTPU_class\paper\code\results_model_inference_summary_0609.csv


,tokenizer,repr,model,prep_sec,feat_sec,train_sec,acc,f1,roc_auc,pr_auc,...,tokenize_mean_sec,feature_mean_sec,predict_mean_sec,total_infer_mean_sec,total_infer_median_sec,total_infer_min_sec,total_infer_max_sec,total_infer_p95_sec,text_length_mean,token_count_mean
0,ckip,fasttext_mean,svm_rbf,204.058941,3.432719,0.188101,0.700361,0.770083,0.903604,0.975405,...,0.154289,0.000842,0.001492,0.156623,0.054578,0.019096,1.384472,0.613301,216.592058,124.620939
1,jieba,tfidf,lr,1.578999,0.445531,0.003299,0.801444,0.862843,0.901745,0.975289,...,0.002117,0.000857,0.000374,0.003349,0.001933,0.000782,0.021140,0.010912,216.592058,122.862816
2,jieba,w2v_mean,svm_rbf,1.578999,1.510796,0.204456,0.747292,0.813830,0.899685,0.975225,...,0.001897,0.000401,0.001166,0.003463,0.001867,0.000880,0.018726,0.011553,216.592058,122.862816
3,ckip,tfidf_svd300,xgb,204.058941,0.710974,2.057627,0.851986,0.908686,0.898109,0.975032,...,0.159542,0.002483,0.001858,0.163884,0.066112,0.025178,1.360378,0.623345,216.592058,124.620939
4,ckip,fasttext_mean,mlp,204.058941,3.432719,0.119805,0.855596,0.912664,0.900695,0.974641,...,0.153402,0.000859,0.001122,0.155383,0.051517,0.017933,1.390679,0.613935,216.592058,124.620939
5,ckip,w2v_mean,svm_rbf,204.058941,1.279816,0.194388,0.736462,0.805333,0.899240,0.974485,...,0.154324,0.000484,0.001488,0.156296,0.052582,0.019289,1.371385,0.618318,216.592058,124.620939
6,ckip,w2v_mean,svm_poly,204.058941,1.279816,0.141978,0.722022,0.791328,0.898109,0.974295,...,0.154044,0.000485,0.001389,0.155919,0.052570,0.018200,1.371695,0.619928,216.592058,124.620939
7,ckip,fasttext_mean,lr,204.058941,3.432719,0.033527,0.765343,0.831169,0.900856,0.974289,...,0.160060,0.000849,0.001108,0.162017,0.058624,0.025085,1.386452,0.616158,216.592058,124.620939
8,ckip,w2v_mean,lr,204.058941,1.279816,0.048457,0.776173,0.842640,0.899887,0.974257,...,0.159484,0.000497,0.001136,0.161118,0.060178,0.024911,1.366395,0.616113,216.592058,124.620939
9,ckip,tfidf,lr,204.058941,0.264325,0.003503,0.812274,0.870647,0.897786,0.973989,...,0.157671,0.001002,0.000423,0.159095,0.057731,0.024303,1.342482,0.606333,216.592058,124.620939


## 10. 建議你正式跑的順序

1. 先把 `BENCHMARK_TEXT_LIMIT = 30` 跑通流程。
2. 確認 CKIP、gensim、xgboost 都能正常執行。
3. 再改成 `BENCHMARK_TEXT_LIMIT = None`、`BENCHMARK_REPEATS = 3` 跑正式版。
4. 正式輸出後，再讓 `plotly_0609.ipynb` 改讀 `results_model_matrix_0609.csv`。

如果你要，我下一步可以直接幫你把 `plotly_0609.ipynb` 改成優先讀 `results_model_matrix_0609.csv`。
